In [52]:
import re
import wikitextparser as wtp
import pandas as pd
import spacy
from cleantext import clean
from alphabet_detector import AlphabetDetector

ad = AlphabetDetector()

nlp = spacy.load("en_core_web_lg")
sent_max = 10

In [120]:
from alphabet_detector import AlphabetDetector
import bisect

def find_non_latin_spans(text):
    # Initialize the AlphabetDetector
    ad = AlphabetDetector()
    
    # Initialize list to store non-Latin spans
    non_latin_spans = []
    
    # Iterate through each character in the text
    start = None
    for i, char in enumerate(text):
        # Check if the character is not in the Latin alphabet
        if not ad.only_alphabet_chars(char, "LATIN"):
            # If this is the start of a new span, record the start index
            if start is None:
                start = i
        # If the character is in the Latin alphabet and we were recording a span, record the end index
        elif start is not None:
            non_latin_spans.append((start, i - 1))  # Subtract 1 from end index to include the Latin character
            start = None
    
    # If the last character was part of a non-Latin span, record the end index
    if start is not None:
        non_latin_spans.append((start, len(text) - 1))
    
    return non_latin_spans

def remove_non_latin_spans(text):
    # Find non-Latin spans
    non_latin_spans = find_non_latin_spans(text)
    
    # Initialize the modified text
    modified_text = ""
    
    # Initialize the start index for the next non-Latin span
    start_index = 0
    
    # Iterate through each non-Latin span
    for span in non_latin_spans:
        # Append the Latin part before the current non-Latin span
        modified_text += text[start_index:span[0]]
        
        # Move the start index to the end of the current non-Latin span
        start_index = span[1] + 1
    
    # Append the remaining Latin part after the last non-Latin span
    modified_text += text[start_index:]
    
    # Remove any extra spaces and characters caused by removing the non-Latin spans
    modified_text = modified_text.strip()
    
    # # Remove unnecessary parentheses
    # modified_text = modified_text.replace("(", "").replace(")", "")
    
    return modified_text


def plain_text(text):
    # plaintext = re.sub(r'[\(\[][\)\]]', '', text)
    # plaintext = re.sub('\xa0', ' ', plaintext) # spacje niełamiące
    plaintext = re.sub(r'[ ]{2}', ' ', text) # spacje wielokrotne
    plaintext = re.sub(r' ([,\.:;])', r'\1', plaintext) # spacje pozostałe po usuniętych templatach
    plaintext = re.sub(r'([,:;])([^\s])', r'\1 \2', plaintext) # brak spacji między końcem zdania (pomijając kropkę, która może być w skrótowcach)
    plaintext = re.sub(r'\([ ,]+', '(', plaintext) # spacje pozostałe po usuniętych templatach
    plaintext = re.sub(r'[ ,]+\)', ')', plaintext) # spacje pozostałe po usuniętych templatach
    plaintext = re.sub(r'\(\)', '', plaintext) # spacje pozostałe po usuniętych templatach
    # plaintext = re.sub(r'([\.,;\"\'\?\[\]\(\)\!\-]{1})[\.,;\"\'\?\[\]\(\)\!\-]+', r'\1', text) # spacje wielokrotne
    plaintext = plaintext.replace('\n\n',' ').strip() # zostawia wyliczenia, np. gdy jest \n* item 1\n* item 2 ...
    # print(plaintext)
    return plaintext

In [150]:
wiki = pd.read_csv('wikipedia_summaries4.csv')
wiki = wiki[['title','text','timestamp','wikitext']].dropna(subset = 'wikitext').reset_index().drop(axis = 1, labels = ['index'])
# wiki_no_dup = wiki[['title','text','timestamp','wikitext']].drop_duplicates(subset = 'title')
wiki1 = pd.read_csv('wikipedia_summaries3.csv')
wiki1 = wiki1[['title','text','timestamp','wikitext']].dropna(subset = 'wikitext').reset_index().drop(axis = 1, labels = ['index'])

wiki_all = pd.concat([wiki1, wiki], ignore_index=True)
wiki_all = wiki_all[['title','text','timestamp','wikitext']].dropna(subset = 'wikitext').reset_index().drop(axis = 1, labels = ['index'])
wiki_no_dup = wiki_all[['title','text','timestamp']].drop_duplicates(subset = 'title')

In [151]:
def clean_all(text):
    cleaned_text = remove_non_latin_spans(text)
    cleaned_text = clean(cleaned_text,fix_unicode=True,to_ascii=True,lower=False,normalize_whitespace=True,no_line_breaks=True,
      strip_lines=True,keep_two_line_breaks=False,no_urls=True,no_emails=False,no_phone_numbers=False,
      no_numbers=False,no_digits=False,no_currency_symbols=False,no_punct=False,no_emoji=False,
      replace_with_url="",replace_with_email="<EMAIL>",replace_with_phone_number="<PHONE>",
      replace_with_number="<NUMBER>",replace_with_digit="0",replace_with_currency_symbol="<CUR>",replace_with_punct="",lang="en")
    cleaned_text = plain_text(cleaned_text)
    return cleaned_text

wiki_no_dup['text'] = wiki_no_dup['text'].apply(lambda x: clean_all(x))

In [155]:
wiki_no_dup.to_csv("wikipedia_summaries_fromAPI.csv", mode='a', header=False)

In [156]:
wiki_no_dup

,title,text,timestamp
0,Americium-241,"Americium-241 (Am, Am-241) is an isotope of am...",2021-08-07T17:17:36Z
1,World_record_progression_100_metres_freestyle,The first world record in the 100 metres frees...,2021-12-28T12:36:24Z
2,Coed_Coch,"Coed Coch, in Dolwen, Conwy, Wales, is a large...",2021-12-30T23:00:25Z
3,Alfred_Hopkinson,Sir Alfred Hopkinson (28 June 1851 - 11 Novemb...,2021-08-15T16:21:20Z
4,Khalifa,Khalifa or Khalifah (Arabic:) is a name or tit...,2021-12-23T02:49:48Z
...,...,...,...
1046,Solar-powered_watch,A solar-powered watch or light-powered watch i...,2020-08-11T12:45:47Z
1047,Fred_Musobo,Fred Musobo (born 12 August 1996) is an Uganda...,2021-10-09T15:24:33Z
1048,Hypomecis_punctinalis,"Hypomecis punctinalis, the pale oak beauty, is...",2021-03-23T14:44:28Z
1049,Battle_Creek_Sanitarium,The Battle Creek Sanitarium was a world-renown...,2021-10-16T02:53:10Z


## Example

In [121]:
# Test the function
text = "ελληνικά means 汉语 greek and 粵語"
non_latin_spans = find_non_latin_spans(text)
if non_latin_spans:
    print("Non-Latin spans found:")
    for span in non_latin_spans:
        print(f"Start index: {span[0]}, End index: {span[1]}, Span: '{text[span[0]:span[1]+1]}'")
else:
    print("No non-Latin spans found.")

# Test the function
text = "ελληνικά means 汉语 greek and 粵語 (example)"
cleaned_text = remove_non_latin_spans(text)
print("Original text:", text)
print("Cleaned text:", cleaned_text)

plain_text("aasg ( )  asgaga () asfag . asavb")

Non-Latin spans found:
Start index: 0, End index: 7, Span: 'ελληνικά'
Start index: 15, End index: 16, Span: '汉语'
Start index: 28, End index: 29, Span: '粵語'
Original text: ελληνικά means 汉语 greek and 粵語 (example)
Cleaned text: means  greek and  (example)


'aasg  asgaga  asfag. asavb'

In [116]:
text = wiki_no_dup.iloc[15].text
# text = wiki_no_dup[wiki_no_dup.title == 'Dharambir_Agnihotri'].iloc[0].text
print(text)
cleaned_text = remove_non_latin_spans(text)
cleaned_text = plain_text(cleaned_text)
cleaned_text = clean(cleaned_text,fix_unicode=True,to_ascii=True,lower=False,normalize_whitespace=True,no_line_breaks=True,
      strip_lines=True,keep_two_line_breaks=False,no_urls=True,no_emails=False,no_phone_numbers=False,
      no_numbers=False,no_digits=False,no_currency_symbols=False,no_punct=False,no_emoji=False,
      replace_with_url="",replace_with_email="<EMAIL>",replace_with_phone_number="<PHONE>",
      replace_with_number="<NUMBER>",replace_with_digit="0",replace_with_currency_symbol="<CUR>",replace_with_punct="",lang="en")
print("\nCleaned text:\n", cleaned_text)

Henri Jibrayel (; born on in Marseille) is a French politician with Lebanese and Assyrian roots. His father was an Assyrian survivor of the Assyrian genocide, that took place in present-day Turkey, who had taken refuge with his parents in a Beirut slum. He married in 1938 a Lebanese Maronite young woman from Bkassine (near Jezzine), then joined the Free French Forces after De Gaulle's Appeal of 18 June. After the war, the family got the French naturalisation and was hosted by its new fatherland in a slum near Marseille. The father was sent in Madagascar till 1950 to repress the anticolonial insurgency, then again in Indochina in one of France's colonial wars. In 1963, the family, including 8 children, tries a comeback in Lebanon, and settle in Ain al-Remmane, but this attempt led to a fiasco, and two years later the family turned back to Marseille, Henri left school at 15, became a crane driver, and afterwards entered the French Poste. After being a trade-unionist at the Poste, he beca

# Śmieci

In [10]:
import pywikibot
from pywikibot import textlib

site = pywikibot.Site('en', 'wikipedia')  # The site we want to run our bot on
textlib.remove_markup(wiki_no_dup.iloc[614].wikitext)